<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ocp22_load_pretrained_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Use OCP22 pretrained models for JARVIS-Leaderboard

In [ ]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-cluster -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

In [ ]:
import os
if not os.path.exists('ocp'):
     !git clone https://github.com/Open-Catalyst-Project/ocp.git
os.chdir('ocp')
!pip install -e .

In [ ]:
pip -q install  lmdb pymatgen==2023.5.10 ase==3.22.1 orjson e3nn==0.2.6 wandb

In [ ]:
import os, torch
from ase.io import read
from ocpmodels.preprocessing import AtomsToGraphs
from ocpmodels.models import SchNet,GemNetOC,GemNetT

In [ ]:
# !wget https://dl.fbaipublicfiles.com/opencatalystproject/models/2022_09/oc22/s2ef/gnoc_oc22_all_s2ef.pt
# !wget https://dl.fbaipublicfiles.com/opencatalystproject/models/2022_09/oc22/s2ef/gndt_oc22_all_s2ef.pt
!wget https://dl.fbaipublicfiles.com/opencatalystproject/models/2023_05/oc22/s2ef/gnoc_oc22_oc20_all_s2ef.pt
# !wget https://dl.fbaipublicfiles.com/opencatalystproject/models/2022_09/oc22/s2ef/gnoc_finetune_all_s2ef.pt


In [ ]:
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
fname = "gnoc_oc22_oc20_all_s2ef.pt" #"gnoc_oc22_all_s2ef.pt"
attrs = torch.load(fname, map_location=device)["config"][
    "model_attributes"
]
weights_path = fname
model = GemNetOC(None, None, 1, **attrs)

ckpt = torch.load(weights_path,map_location=torch.device('cpu'))

state_dict = ckpt["state_dict"]
state_dict = {k[2 * len("module.") :]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict, strict=True)
model.to(device)